# Study 1 Analysis
Code for analyzing data from the survey in Study 1

# Load packages

In [ ]:
import json
import os
import math
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.options.mode.chained_assignment = None  # default='warn'

# Load Data

In [ ]:
survey_data_df = pd.read_csv(
    "./survey-data/final-survey-data_04-13-25.csv",
    keep_default_na=False,
)
print(json.dumps(list(survey_data_df.columns), indent=4))
survey_data_df.head()

In [ ]:
confidence_col = "Again, think about the AI tool you use most to caption products. When the tool says a photo is not good enough to caption or returns a similar error, how confident are you in knowing why the photo is not good enough?"


survey_data_df[survey_data_df[confidence_col] != ""][confidence_col].map(
    {
        "Extremely confident": 5,
        "Very confident": 4,
        "Somewhat confident": 3,
        "Slightly confident": 2,
        "Not at all confident": 1,
    }
).describe()

# Descriptive Stats

In [ ]:
def get_descriptive_stats(df, col_dict, answer_dict, show_percent=False):
    """
    Generate a descriptive statistics DataFrame for scenario-based questions.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        col_dict (dict): Mapping of original column names to display names.
        answer_dict (dict): Mapping of answer text to values (order).
        show_percent (bool): Whether to include percentages in the output.

    Returns:
        pd.DataFrame: Descriptive statistics table.
    """
    result = pd.DataFrame(
        {x: "" for x in ["Answer"] + list(col_dict.values())}, index=[]
    )
    total_responses = len(df)
    for answer in answer_dict.keys():
        current_row = {"Answer": answer}
        for col in col_dict.keys():
            count = len(df[df[col] == answer])
            if show_percent:
                percent = count / total_responses if total_responses > 0 else 0
                current_row[col_dict[col]] = f"{count} ({percent:.1%})"
            else:
                current_row[col_dict[col]] = count
        result = pd.concat([result, pd.DataFrame([current_row])], ignore_index=True)

    return result

## Preferences for AI vs Human Assistance

In [ ]:
def plot_ai_human_side_by_side(
    left_df,
    right_df,
    left_title="Left",
    right_title="Right",
    answer_order=None,
    colors=None,
    wrap_width_left=20,
    wrap_width_right=20,
    min_label_threshold=5,
    figsize=(14, 5),
    font_family="sans-serif",
    font_size=11,
    center_label=None,
    share_xlim=True,
    xlim=None,
    legend_ncol=3,
    sort_answers=None,
):
    """
    Draw two diverging stacked horizontal bar charts side-by-side for comparison,
    each with its own y-axis labels and title, and a shared legend underneath.

    The shared legend is ordered so that it reads left-to-right across rows,
    e.g., with legend_ncol=3, the first row shows the first three categories
    in order, and the remainder appear on the second row.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=figsize)

    # Left subplot
    plot_ai_human_diverging_bars(
        left_df,
        answer_order=answer_order,
        colors=colors,
        title=left_title,
        wrap_width=wrap_width_left,
        min_label_threshold=min_label_threshold,
        figsize=figsize,
        font_family=font_family,
        font_size=font_size,
        center_label=center_label,
        ax=ax_l,
        sort_answers=sort_answers,
    )

    # Right subplot
    plot_ai_human_diverging_bars(
        right_df,
        answer_order=answer_order,
        colors=colors,
        title=right_title,
        wrap_width=wrap_width_right,
        min_label_threshold=min_label_threshold,
        figsize=figsize,
        font_family=font_family,
        font_size=font_size,
        center_label=center_label,
        ax=ax_r,
        sort_answers=sort_answers,
    )

    # Harmonize x-limits if requested
    if xlim is not None:
        ax_l.set_xlim(*xlim)
        ax_r.set_xlim(*xlim)
    elif share_xlim:
        l0, l1 = ax_l.get_xlim()
        r0, r1 = ax_r.get_xlim()
        max_abs = max(abs(l0), abs(l1), abs(r0), abs(r1))
        ax_l.set_xlim(-max_abs, max_abs)
        ax_r.set_xlim(-max_abs, max_abs)

    # Build legend handles/labels explicitly to control order (row-wise left-to-right)
    if colors is None:
        colors = [
            "#4B8BBE",
            "#89B4E0",
            "#E5E5E5",
            "#F5B97D",
            "#E07A5F",
        ]
    if answer_order is None:
        answer_order = [
            "Almost always just an AI based tool",
            "Most often just an AI based tool",
            "Both an AI based tool and human sighted assistance",
            "Most often just human sighted assistance",
            "Almost always just human sighted assistance",
        ]
    handles = [
        mpatches.Patch(color=color, label=label)
        for color, label in zip(colors, answer_order)
    ]
    labels = answer_order

    # Shared legend underneath; Matplotlib fills row-wise left-to-right across ncol
    ncols = legend_ncol if legend_ncol and legend_ncol > 0 else len(labels)

    # Hack for reordering
    if ncols != len(labels):
        new_handles = [
            handles[0],
            handles[3],
            handles[1],
            handles[4],
            handles[2],
            handles[3],
        ]
        new_labels = [
            labels[0],
            labels[3],
            labels[1],
            labels[4],
            labels[2],
        ]
    else:
        new_handles = handles
        new_labels = labels
    # Add a thin black border to each legend patch (use facecolor, not color, to avoid warning)
    bordered_handles = [
        mpatches.Patch(
            facecolor=patch.get_facecolor(),
            label=patch.get_label(),
            edgecolor="black",
            linewidth=0.1,
        )
        for patch in new_handles
    ]
    fig.legend(
        bordered_handles,
        new_labels,
        loc="lower center",
        ncol=ncols,
        frameon=False,
        bbox_to_anchor=(0.5, 0),
        fontsize=font_size - 1,
    )

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.15)
    return fig, (ax_l, ax_r)


def plot_ai_human_diverging_bars(
    dataframe,
    answer_order=None,
    colors=None,
    title="Distribution of AI vs Human Assistance Preferences",
    wrap_width=20,
    min_label_threshold=5,
    figsize=(8, 5),
    font_family="sans-serif",
    font_size=10,
    center_label=None,
    ax=None,
    sort_answers=None,
):
    """
    Render a diverging stacked horizontal bar chart for AI vs Human assistance preferences.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Either a wide DataFrame where the index are y-axis labels (scenarios) and columns
        are the five answer categories, or a long DataFrame containing a column named
        "Answer" with one row per category (in which case it will be pivoted via .set_index('Answer').T).
    answer_order : list[str] | None
        The ordered list of five category names from most-AI to most-human with the
        "Both" category in the middle. Defaults to the study's categories.
    colors : list[str] | None
        Five hex color codes matching answer_order. Defaults to the study's palette.
    title : str
        Plot title.
    wrap_width : int
        Number of characters to wrap y-axis labels to.
    min_label_threshold : int
        Minimum bar width (in count units) to draw a value label.
    figsize : tuple[int, int]
        Figure size for matplotlib.
    font_family : str
        Matplotlib font family to apply.
    font_size : int
        Base font size.
    center_label : str | None
        Optional override for the middle ("Both") label in the legend; defaults to answer_order[2].
    ax : matplotlib.axes.Axes | None
        Existing Axes to draw on; if None, a new Figure/Axes is created.

    Returns
    -------
    (fig, ax)
        Matplotlib Figure and Axes.
    """
    import matplotlib.pyplot as plt
    import textwrap
    from matplotlib.ticker import MaxNLocator, FuncFormatter

    # Defaults
    if answer_order is None:
        answer_order = [
            "Almost always just an AI based tool",
            "Most often just an AI based tool",
            "Both an AI based tool and human sighted assistance",
            "Most often just human sighted assistance",
            "Almost always just human sighted assistance",
        ]
    if colors is None:
        colors = [
            "#4B8BBE",  # blue: Almost always just an AI based tool
            "#89B4E0",  # light blue: Most often just an AI based tool
            "#E5E5E5",  # gray: Both an AI based tool and human sighted assistance
            "#F5B97D",  # light orange: Most often just human sighted assistance
            "#E07A5F",  # orange: Almost always just human sighted assistance
        ]
    if center_label is None:
        center_label = answer_order[2]

    # Prepare data (allow either long with "Answer" or wide with columns as categories)
    if "Answer" in dataframe.columns:
        plot_df = dataframe.set_index("Answer").T
    else:
        plot_df = dataframe.copy()

    # Validate and reorder columns to match answer_order
    missing = [a for a in answer_order if a not in plot_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    plot_df = plot_df[answer_order]

    # Reverse y order to match prior convention
    plot_df = plot_df.iloc[::-1]

    # Sort scenarios by AI-leaning totals (then flipped as in the original)
    if sort_answers is not None:
        ai_pref_totals = None
        for answer in sort_answers:
            if ai_pref_totals is None:
                ai_pref_totals = plot_df[answer].astype(int)
            else:
                ai_pref_totals = ai_pref_totals + plot_df[answer].astype(int)
        sorted_indices = ai_pref_totals.sort_values(ascending=False).index[::-1]
        plot_df_sorted = plot_df.loc[sorted_indices]
    else:
        plot_df_sorted = plot_df

    # Y labels (wrapped)
    y_labels_sorted_wrapped = [
        "\n".join(textwrap.wrap(label, wrap_width))
        for label in plot_df_sorted.index.tolist()
    ]

    # Values
    v0 = plot_df_sorted[answer_order[0]].astype(int).values
    v1 = plot_df_sorted[answer_order[1]].astype(int).values
    v2 = plot_df_sorted[answer_order[2]].astype(int).values
    v3 = plot_df_sorted[answer_order[3]].astype(int).values
    v4 = plot_df_sorted[answer_order[4]].astype(int).values

    # Left offsets to center the middle (Both) bin at 0
    left0 = -v1 - v2 / 2 - v0
    left1 = -v2 / 2 - v1
    left2 = -v2 / 2
    left3 = v2 / 2
    left4 = v3 + v2 / 2

    # Create axes if needed
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # Global style tweaks
    plt.rcParams["font.family"] = font_family
    plt.rcParams["font.size"] = font_size

    # Draw bars
    bars = []
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v0,
            left=left0,
            color=colors[0],
            label=answer_order[0],
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v1,
            left=left1,
            color=colors[1],
            label=answer_order[1],
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v2,
            left=left2,
            color=colors[2],
            label=center_label,
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v3,
            left=left3,
            color=colors[3],
            label=answer_order[3],
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v4,
            left=left4,
            color=colors[4],
            label=answer_order[4],
            zorder=3,
        )
    )

    # Bar labels (only for sufficiently wide segments)
    for group in bars:
        for bar in group:
            width = bar.get_width()
            if abs(width) > min_label_threshold:
                x = bar.get_x() + width / 2
                y = bar.get_y() + bar.get_height() / 2
                ax.text(
                    x,
                    y,
                    f"{int(abs(width))}",
                    va="center",
                    ha="center",
                    color="black",
                    fontsize=font_size - 1,
                    zorder=4,
                    fontweight="semibold",
                )

    # Formatting
    ax.set_xlabel("")
    ax.set_ylabel("")
    if title:
        ax.set_title(
            textwrap.fill(title, width=40),
            fontsize=font_size + 1,
            fontweight="semibold",
        )

    ax.axvline(0, color="black", linewidth=0.8, zorder=1)
    ax.grid(axis="x", linestyle="-", zorder=2)

    # Symmetric x-limits and absolute-value tick labels
    # make them the max rounded to the nearest ceiling 5
    max_val = math.ceil(max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1])) / 5) * 5
    ax.set_xlim(-max_val, max_val)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(abs(x))}"))

    # remove spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)

    fig.tight_layout()
    return fig, ax

In [ ]:
answer_dict = {
    "Almost always just human sighted assistance": 2,
    "Most often just human sighted assistance": 1,
    "Both an AI based tool and human sighted assistance": 0,
    "Most often just an AI based tool": -1,
    "Almost always just an AI based tool": -2,
}
scenario_cols = {
    "Which of the following are you most likely to use when searching for a specific product item at a physical store?": "searching for product physical store",
    "Which of the following are you most likely to use when browsing products at a physical store?": "browsing products at store",
    "Which of the following are you most likely to use when wanting to compare the details of two products side by side?": "comparing details of two products",
    "Which of the following are you most likely to use when identifying an unknown item in your home?": "id. unknown item in home",
    "Which of the following are you most likely to use when reading a label on a food item?": "reading food label",
    "Which of the following are you most likely to use when reading a label on medication?": "reading label on medication",
    "Which of the following are you most likely to use when identifying personal care products or toiletries?": "id. personal products / toiletries",
    "Which of the following are you most likely to use when checking expiration dates on products?": "checking expiration dates",
    "Which of the following are you most likely to use when checking allergen information on products?": "checking product allergen info",
}
print("Which of the following are you most likely to use when X is your top concern?")
display(
    get_descriptive_stats(
        survey_data_df[scenario_cols.keys()],
        scenario_cols,
        answer_dict,
        show_percent=True,
    )
)

concern_cols = {
    "Which of the following are you most likely to use when efficiency in identifying and understanding a product is your top concern?": "efficiency",
    "Which of the following are you most likely to use when data privacy in identifying and understanding a product is your top concern (e.g., the extent to which data you share is kept private)?": "data privacy",
    "Which of the following are you most likely to use when personal privacy in identifying and understanding a product is your top concern (e.g., for fear of embarrassment or stigma)?": "personal privacy",
    "Which of the following are you most likely to use when accuracy in identifying and understanding a product is your top concern?": "accuracy",
    "Which of the following are you most likely to use when financial cost in identifying and understanding a product is your top concern?": "financial cost",
    "Which of the following are you most likely to use when safety in identifying and understanding a product is your top concern?": "safety",
}
print("Which of the following are you most likely to use when X is your top concern?")
display(
    get_descriptive_stats(
        survey_data_df[concern_cols.keys()],
        concern_cols,
        answer_dict,
        show_percent=True,
    )
)

In [ ]:
# Left/right can be long (has "Answer") or wide (columns == 5 categories)
fig, (ax_l, ax_r) = plot_ai_human_side_by_side(
    left_df=get_descriptive_stats(
        survey_data_df[scenario_cols.keys()],
        scenario_cols,
        answer_dict,
        show_percent=False,
    ),
    right_df=get_descriptive_stats(
        survey_data_df[concern_cols.keys()],
        concern_cols,
        answer_dict,
        show_percent=False,
    ),
    left_title=f"Scenario-Based Preference for AI vs Human Assistance (N = {len(survey_data_df[['Timestamp'] + list(scenario_cols.keys())]['Timestamp'].unique())})",
    right_title=f"Concern-Based Preference for AI vs Human Assistance (N = {len(survey_data_df[['Timestamp'] + list(concern_cols.keys())]['Timestamp'].unique())})",
    figsize=(12, 6),
    font_size=11,
    colors=[
        "#ca0020",  # blue: Almost always just an AI based tool
        "#f4a582",  # light blue: Most often just an AI based tool
        "#f7f7f7",  # gray: Both an AI based tool and human sighted assistance
        "#92c5de",  # light orange: Most often just human sighted assistance
        "#0571b0",  # orange: Almost always just human sighted assistance
    ],
    wrap_width_left=21,
    wrap_width_right=10,
    sort_answers=[
        "Both an AI based tool and human sighted assistance",
        "Most often just an AI based tool",
        "Almost always just an AI based tool",
    ],
)
# save as pdf
os.makedirs("./plots", exist_ok=True)
fig.savefig(
    "./plots/scenario-vs-concern-preference.pdf", bbox_inches="tight", pad_inches=0
)

## Image Quality Problems and Assessment

In [ ]:
def plot_divering_bars_side_by_side(
    left_df,
    right_df,
    left_title="Left",
    right_title="Right",
    answer_order=None,
    colors=None,
    wrap_width_left=20,
    wrap_width_right=20,
    min_label_threshold=5,
    figsize=(14, 5),
    font_family="sans-serif",
    font_size=11,
    center_label=None,
    share_xlim=True,
    xlim=None,
    legend_ncol=3,
    sort_answers=None,
):
    """
    Draw two diverging stacked horizontal bar charts side-by-side for comparison,
    each with its own y-axis labels and title, and a shared legend underneath.

    The shared legend is ordered so that it reads left-to-right across rows,
    e.g., with legend_ncol=3, the first row shows the first three categories
    in order, and the remainder appear on the second row.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=figsize)

    # Left subplot
    plot_diverging_even_bars(
        left_df,
        answer_order=answer_order,
        colors=colors,
        title=left_title,
        wrap_width=wrap_width_left,
        min_label_threshold=min_label_threshold,
        figsize=figsize,
        font_family=font_family,
        font_size=font_size,
        ax=ax_l,
        sort_answers=sort_answers,
    )

    # Right subplot
    plot_diverging_even_bars(
        right_df,
        answer_order=answer_order,
        colors=colors,
        title=right_title,
        wrap_width=wrap_width_right,
        min_label_threshold=min_label_threshold,
        figsize=figsize,
        font_family=font_family,
        font_size=font_size,
        ax=ax_r,
        sort_answers=sort_answers,
    )

    # Harmonize x-limits if requested
    if xlim is not None:
        ax_l.set_xlim(*xlim)
        ax_r.set_xlim(*xlim)
    elif share_xlim:
        l0, l1 = ax_l.get_xlim()
        r0, r1 = ax_r.get_xlim()
        max_abs = max(abs(l0), abs(l1), abs(r0), abs(r1))
        ax_l.set_xlim(-max_abs, max_abs)
        ax_r.set_xlim(-max_abs, max_abs)

    # Build legend handles/labels explicitly to control order (row-wise left-to-right)
    if colors is None:
        colors = [
            "#4B8BBE",
            "#89B4E0",
            "#E5E5E5",
            "#F5B97D",
            "#E07A5F",
        ]
    if answer_order is None:
        answer_order = [
            "Almost always just an AI based tool",
            "Most often just an AI based tool",
            "Both an AI based tool and human sighted assistance",
            "Most often just human sighted assistance",
            "Almost always just human sighted assistance",
        ]
    handles = [
        mpatches.Patch(color=color, label=label)
        for color, label in zip(colors, answer_order)
    ]
    labels = answer_order

    # Shared legend underneath; Matplotlib fills row-wise left-to-right across ncol
    ncols = legend_ncol if legend_ncol and legend_ncol > 0 else len(labels)

    # Hack for reordering
    # if ncols != len(labels):
    #     new_handles = [
    #         handles[0],
    #         handles[3],
    #         handles[1],
    #         handles[4],
    #         handles[2],
    #         handles[3],
    #     ]
    #     new_labels = [
    #         labels[0],
    #         labels[3],
    #         labels[1],
    #         labels[4],
    #         labels[2],
    #     ]
    # else:
    new_handles = handles
    new_labels = labels
    # Add a thin black border to each legend patch (use facecolor, not color, to avoid warning)
    bordered_handles = [
        mpatches.Patch(
            facecolor=patch.get_facecolor(),
            label=patch.get_label(),
            edgecolor="black",
            linewidth=0.1,
        )
        for patch in new_handles
    ]
    fig.legend(
        bordered_handles,
        new_labels,
        loc="lower center",
        ncol=ncols,
        frameon=False,
        bbox_to_anchor=(0.5, 0),
        fontsize=font_size - 1,
    )

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.15)
    return fig, (ax_l, ax_r)


def plot_diverging_even_bars(
    dataframe,
    answer_order,
    colors=None,
    title="",
    wrap_width=20,
    min_label_threshold=5,
    figsize=(8, 5),
    font_family="sans-serif",
    font_size=10,
    ax=None,
    sort_answers=None,
    left_from_center=True,
    legend=False,
):
    """
    Plot a diverging stacked horizontal bar chart with an EVEN number of categories.
    Half of the categories are stacked to the left of zero, half to the right.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Either a wide DataFrame where the index are y-axis labels and columns are categories,
        or a long DataFrame with a column named "Answer" which will be pivoted via
        .set_index('Answer').T.
    answer_order : list[str]
        Ordered list of category names of EVEN length. The first half are the left-side
        categories, the second half are the right-side categories. Within each half,
        the order should be from nearest-to-zero outward if left_from_center is True
        (recommended for nice layering from center), else from far edge to center.
    colors : list[str] | None
        List of hex colors (same length as answer_order). If None, a default palette is used.
    title : str
        Plot title.
    wrap_width : int
        Line-wrap width for y-axis labels.
    min_label_threshold : int
        Minimum bar width to draw a numeric segment label.
    figsize : tuple[int, int]
        Figure size if ax is not provided.
    font_family : str
        Matplotlib font family.
    font_size : int
        Base font size.
    ax : matplotlib.axes.Axes | None
        Axes to draw on; creates a new one if None.
    sort_answers : list[str] | None
        If provided, rows (y labels) are sorted descending by the sum of these columns.
    left_from_center : bool
        If True (default), stacks left and right halves starting from the center outwards.

    Returns
    -------
    (fig, ax)
        Matplotlib Figure and Axes.
    """
    import matplotlib.pyplot as plt
    import textwrap
    from matplotlib.ticker import MaxNLocator, FuncFormatter

    if len(answer_order) % 2 != 0:
        raise ValueError("answer_order must have an even number of categories")

    if colors is None:
        # Fallback palette sized to answer_order
        default_colors = [
            "#4B8BBE",
            "#89B4E0",
            "#E5E5E5",
            "#F5B97D",
            "#E07A5F",
            "#9C6ADE",
            "#5DC0A6",
            "#FFC857",
        ]
        # Repeat or trim to match length
        repeats = int(np.ceil(len(answer_order) / len(default_colors)))
        colors = (default_colors * repeats)[: len(answer_order)]

    # Prepare data
    if "Answer" in dataframe.columns:
        plot_df = dataframe.set_index("Answer").T
    else:
        plot_df = dataframe.copy()

    # Validate columns
    missing = [a for a in answer_order if a not in plot_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep only in specified order
    plot_df = plot_df[answer_order]

    # Reverse y order for consistency with earlier visuals
    plot_df = plot_df.iloc[::-1]

    # Optional sort of rows by selected answer columns
    if sort_answers:
        totals = None
        for name in sort_answers:
            col = plot_df[name].astype(int)
            totals = col if totals is None else totals + col
        sorted_idx = totals.sort_values(ascending=False).index[::-1]
        plot_df = plot_df.loc[sorted_idx]

    # Wrapped y labels
    y_labels = [
        "\n".join(textwrap.wrap(lbl, wrap_width)) for lbl in plot_df.index.tolist()
    ]

    # Split categories
    half = len(answer_order) // 2
    left_cats = answer_order[:half]
    right_cats = answer_order[half:]

    # Order within halves
    if left_from_center:
        # Draw from the center outward: left half reversed so nearest-to-center first
        left_seq = list(reversed(left_cats))  # near center -> outward
        right_seq = right_cats  # near center -> outward
    else:
        # Draw from the outside toward center
        left_seq = left_cats  # outward -> center
        right_seq = list(reversed(right_cats))

    # Extract values
    left_values = [plot_df[c].astype(int).values for c in left_seq]
    right_values = [plot_df[c].astype(int).values for c in right_seq]

    # Compute left offsets (stacking from center to left)
    # For the first left segment (nearest center), start at -width
    left_offsets = []
    cumulative = np.zeros(len(plot_df), dtype=int)
    for idx, w in enumerate(left_values):
        cumulative = cumulative + w
        left_offsets.append(-cumulative)

    # Compute right offsets (stacking from center to right)
    right_offsets = []
    cumulative = np.zeros(len(plot_df), dtype=int)
    for idx, w in enumerate(right_values):
        right_offsets.append(cumulative)  # start at current positive cumulative
        cumulative = cumulative + w

    # Create axes if needed
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # Styles
    plt.rcParams["font.family"] = font_family
    plt.rcParams["font.size"] = font_size

    # Draw left bars (use colors aligned to answer_order, reindexed to left_seq)
    bars = []
    for i, (vals, left) in enumerate(zip(left_values, left_offsets)):
        color = colors[answer_order.index(left_seq[i])]
        bars.append(
            ax.barh(
                y_labels,
                vals,
                left=left,
                color=color,
                label=left_seq[i],
                zorder=3,
            )
        )
    bars = bars[::-1]
    # Draw right bars
    for i, (vals, left) in enumerate(zip(right_values, right_offsets)):
        color = colors[answer_order.index(right_seq[i])]
        bars.append(
            ax.barh(
                y_labels,
                vals,
                left=left,
                color=color,
                label=right_seq[i],
                zorder=3,
            )
        )

    # Labels on segments
    for group in bars:
        for bar in group:
            width = bar.get_width()
            if abs(width) > min_label_threshold:
                x = bar.get_x() + width / 2
                y = bar.get_y() + bar.get_height() / 2
                ax.text(
                    x,
                    y,
                    f"{int(abs(width))}",
                    va="center",
                    ha="center",
                    color="black",
                    fontsize=font_size - 1,
                    zorder=4,
                    fontweight="semibold",
                )

    # Formatting
    ax.set_xlabel("")
    ax.set_ylabel("")
    if title:
        ax.set_title(
            textwrap.fill(title, width=40),
            fontsize=font_size + 1,
            fontweight="semibold",
        )

    ax.axvline(0, color="black", linewidth=0.8, zorder=1)
    ax.grid(axis="x", linestyle="-", zorder=2)

    # Symmetric x-limits and absolute-value tick labels
    left_totals = (
        np.sum(np.column_stack(left_values), axis=1)
        if left_values
        else np.zeros(len(plot_df))
    )
    right_totals = (
        np.sum(np.column_stack(right_values), axis=1)
        if right_values
        else np.zeros(len(plot_df))
    )
    max_side = int(max(left_totals.max(), right_totals.max()))

    import math as _math

    max_val = _math.ceil(max_side / 5) * 5 if max_side > 0 else 5
    ax.set_xlim(-max_val, max_val)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(abs(x))}"))

    # optionally include legend
    if legend:
        import matplotlib.patches as mpatches

        # Extract colors from the bar containers
        colors = []
        for bar_container in bars:
            if hasattr(bar_container, "patches"):
                # Get color from first patch in the container
                colors.append(bar_container.patches[0].get_facecolor())
            else:
                # Fallback color if we can't extract it
                colors.append("gray")

        bordered_handles = [
            mpatches.Patch(
                facecolor=color,
                label=label,
                edgecolor="black",
                linewidth=0.1,
            )
            for color, label in zip(colors, answer_order)
        ]
        fig.legend(
            bordered_handles,
            answer_order,
            loc="lower center",
            ncol=len(answer_order),
            frameon=False,
            bbox_to_anchor=(0.5, -0.1),
            fontsize=font_size - 1,
        )
    # Clean spines
    for spine in ["top", "right", "bottom", "left"]:
        ax.spines[spine].set_visible(False)

    fig.tight_layout()
    return fig, ax

In [ ]:
def compute_descriptive_stats(df, col_dict, answer_dict, exclude_not_sure=False):
    """
    Generate a descriptive statistics DataFrame for scenario-based questions.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        col_dict (dict): Mapping of original column names to display names.
        answer_dict (dict): Mapping of answer text to values (order).

    Returns:
        pd.DataFrame: Descriptive statistics table.
    """
    # make a deep copy of the dataframe
    df = df.copy()

    result = pd.DataFrame(
        {x: "" for x in ["Answer"] + list(col_dict.values())}, index=[]
    )

    # map answers to numbers
    for col in col_dict.keys():
        df[col] = df[col].map(answer_dict)

    # compute average and standard deviation
    for score in ["Average", "Standard Deviation"]:
        current_row = {"Answer": score}
        for col in col_dict.keys():
            curr_data_col = df[col].copy()
            if exclude_not_sure:
                curr_data_col = curr_data_col[
                    curr_data_col > answer_dict["I am not sure"]
                ]
            current_row[col_dict[col]] = round(
                (
                    curr_data_col.mean(skipna=True)
                    if score == "Average"
                    else curr_data_col.std(skipna=True)
                ),
                2,
            )
        result = pd.concat([result, pd.DataFrame([current_row])], ignore_index=True)
    return result

In [ ]:
answer_dict = {
    "To a great extent": 4,
    "Somewhat": 3,
    "Very little": 2,
    "Not at all": 1,
    "I am not sure": 0,
}

In [ ]:
image_quality_cols = {
    "Keeping the tool you use most in mind, how much does lighting in your environment affect the quality of AI generated captions for products?": "lighting",
    "How much does taking clear, non-blurry photos affect the quality of AI generated captions for products?": "blur",
    "How much does having the object in full view of your camera, rather than a partial view of the object, affect the quality of AI generated captions for products?": "framing",
    "How much does rotating your camera or the object affect the quality of AI generated captions for products?": "rotation",
    "How much does your hand placement or how you are holding the object affect the quality of AI generated captions for products?": "hand position",
    "How much does moving your camera closer or further from an object affect the quality of AI generated captions for products?": "distance",
    # "How much does a product being uncommon or unique affect the quality of AI generated captions?": "uniqueness",
}
print("How much does X affect the quality of AI generated captions for products?")
display(
    get_descriptive_stats(
        survey_data_df[image_quality_cols.keys()],
        image_quality_cols,
        answer_dict,
        show_percent=True,
    )
)
display(
    compute_descriptive_stats(
        survey_data_df[image_quality_cols.keys()],
        image_quality_cols,
        answer_dict,
        exclude_not_sure=True,
    )
)

image_assessment_factor_cols = {
    "How well does the AI tool help you assess lighting conditions when taking a photo?": "lighting",
    "How well does the AI tool help you assess whether the photo you took is clear and non-blurry?": "blur",
    "How well does the AI tool help you assess whether an object is in full view of your camera when taking photos?": "framing",
    "How well does the AI tool help you assess object orientation when taking photos?": "rotation",
    "How well does the AI tool help you understand your hand positioning or placement relative to the object you want to photograph?": "hand position",
    "How well does the AI tool help you assess the distance between your camera and the object of interest when taking photos?": "distance",
    # "How well does the AI tool help you assess whether a product you are taking a photo of is uncommon or unique?": "uniqueness",
}
print("How well does AI help assess X when taking a photo?")
display(
    get_descriptive_stats(
        survey_data_df[image_assessment_factor_cols.keys()],
        image_assessment_factor_cols,
        answer_dict,
        show_percent=True,
    )
)
display(
    compute_descriptive_stats(
        survey_data_df[image_assessment_factor_cols.keys()],
        image_assessment_factor_cols,
        answer_dict,
        exclude_not_sure=True,
    )
)

In [ ]:
fig, (ax_l, ax_r) = plot_divering_bars_side_by_side(
    left_df=get_descriptive_stats(
        survey_data_df[image_quality_cols.keys()],
        image_quality_cols,
        answer_dict,
        show_percent=False,
    ),
    right_df=get_descriptive_stats(
        survey_data_df[image_assessment_factor_cols.keys()],
        image_assessment_factor_cols,
        answer_dict,
        show_percent=False,
    ),
    left_title=f"Impact of Image Quality on Caption Quality (N = {len(survey_data_df[['Timestamp'] + list(image_quality_cols.keys())]['Timestamp'].unique())})",
    right_title=f"Ability to Assess Image Quality Issue Impact on Caption (N = {len(survey_data_df[['Timestamp'] + list(image_assessment_factor_cols.keys())]['Timestamp'].unique())})",
    figsize=(12, 4),
    font_size=11,
    colors=[
        "#ca0020",  # not at all
        "#f4a582",  # very little
        "#92c5de",  # somewhat
        "#0571b0",  # to a great extent
    ],
    wrap_width_left=10,
    wrap_width_right=10,
    answer_order=list(answer_dict.keys())[::-1][1:],
    legend_ncol=4,
    sort_answers=["To a great extent", "Somewhat"],
)
# save as pdf
os.makedirs("./plots", exist_ok=True)
fig.savefig(
    "./plots/impact-quality-vs-ability-assess.pdf", bbox_inches="tight", pad_inches=0
)

## Percieved Frequency of Errors

In [ ]:
answer_dict = {
    "Very frequently": 6,
    "Frequently": 5,
    "Occasionally": 4,
    "Rarely": 3,
    "Very rarely": 2,
    "Never": 1,
}

In [ ]:
freq_errs_cols = {
    "Based on your experience, how often are AI generated captions for products not accurate (e.g., captions a box of cereal as pasta)?": "not accurate",
    "Based on your experience, how often are AI generated captions for products accurate but missing critical information (e.g., captions it as a can of beans but does not say what type of beans)?": "missing critical information",
    "Based on your experience, how often are AI generated captions only partially correct (e.g., a can of soup is captioned as a can of green beans)?": "partially correct",
    "Based on your experience, how often do AI generated captions include extra, incorrect details (e.g., a bag of green peas is captioned as a bag of green peas and corn)?": "extra, incorrect details",
    "Based on your experience, how often are AI generated captions completely made up (e.g., you accidentally cover the camera with your finger but AI says it's a picture of the moon)?": "completely made up",
}
print("Based on your experience, how often are AI generated captions for products X?")
display(
    get_descriptive_stats(
        survey_data_df[freq_errs_cols.keys()],
        freq_errs_cols,
        answer_dict,
        show_percent=True,
    )
)
display(
    compute_descriptive_stats(
        survey_data_df[freq_errs_cols.keys()],
        freq_errs_cols,
        answer_dict,
        exclude_not_sure=False,
    )
)

In [ ]:
fig, ax = plot_diverging_even_bars(
    get_descriptive_stats(
        survey_data_df[freq_errs_cols.keys()],
        freq_errs_cols,
        answer_dict,
        show_percent=False,
    ),
    list(answer_dict.keys())[::-1],
    colors=[
        "#b2182b",  # not at all
        "#ef8a62",  # very little
        "#fddbc7",  # somewhat
        "#d1e5f0",  # to a great extent
        "#67a9cf",  # to a great extent
        "#2166ac",  # to a great extent
    ],
    title=f"Perceived Frequency of Errors (N = {len(survey_data_df[['Timestamp'] + list(freq_errs_cols.keys())]['Timestamp'].unique())})",
    wrap_width=20,
    min_label_threshold=5,
    figsize=(9, 3.33333335),
    font_family="sans-serif",
    font_size=10,
    ax=None,
    sort_answers=["Occasionally", "Frequently", "Very frequently"],
    left_from_center=True,
    legend=True,
)
# save as pdf
os.makedirs("./plots", exist_ok=True)
fig.savefig("./plots/perceived-frequency-errors.pdf", bbox_inches="tight", pad_inches=0)

# Statistical analysis 

In [ ]:
image_quality_factor_cols = [
    "Keeping the tool you use most in mind, how much does lighting in your environment affect the quality of AI generated captions for products?",
    "How much does taking clear, non-blurry photos affect the quality of AI generated captions for products?",
    "How much does having the object in full view of your camera, rather than a partial view of the object, affect the quality of AI generated captions for products?",
    "How much does rotating your camera or the object affect the quality of AI generated captions for products?",
    "How much does your hand placement or how you are holding the object affect the quality of AI generated captions for products?",
    "How much does moving your camera closer or further from an object affect the quality of AI generated captions for products?",
    "How much does a product being uncommon or unique affect the quality of AI generated captions?",
]

image_assessment_factor_cols = [
    "How well does the AI tool help you assess lighting conditions when taking a photo?",
    "How well does the AI tool help you assess whether the photo you took is clear and non-blurry?",
    "How well does the AI tool help you assess whether an object is in full view of your camera when taking photos?",
    "How well does the AI tool help you assess object orientation when taking photos?",
    "How well does the AI tool help you understand your hand positioning or placement relative to the object you want to photograph?",
    "How well does the AI tool help you assess the distance between your camera and the object of interest when taking photos?",
    "How well does the AI tool help you assess whether a product you are taking a photo of is uncommon or unique?",
]


mapping_dict = {
    "To a great extent": 5,
    "Somewhat": 4,
    "Very little": 3,
    "Not at all": 2,
    "I am not sure": 1,
}

In [ ]:
condition_col = "Which of the following AI based tools do you use the most to identify and understand products?"
survey_data_df[condition_col].value_counts()

In [ ]:
# calculate mean and statistical differences for how much image quality affects response
for question in image_quality_factor_cols:
    # create dataframes for each condition
    be_my_ai_products_df = survey_data_df[
        survey_data_df[condition_col]
        == "Be My AI (part of Be My Eyes that does not involve human visual intepreters)"
    ]
    seeing_ai_products_df = survey_data_df[
        survey_data_df[condition_col] == "Microsoft Seeing AI"
    ]

    # convert to number
    seeing_ai_products_df[question] = seeing_ai_products_df[question].map(mapping_dict)
    be_my_ai_products_df[question] = be_my_ai_products_df[question].map(mapping_dict)

    # remove rows where any of the image quality factor columns are empty
    seeing_ai_products_df = seeing_ai_products_df[
        seeing_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
    ]
    be_my_ai_products_df = be_my_ai_products_df[
        be_my_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
    ]

    print(
        f"Be My AI: n = {len(be_my_ai_products_df)} | Seeing AI: n = {len(seeing_ai_products_df)}"
    )

    U, p = mannwhitneyu(
        seeing_ai_products_df[question],
        be_my_ai_products_df[question],
        nan_policy="omit",
    )
    print(f"{question}")
    print(
        f"Mean Seeing AI: {seeing_ai_products_df[question].mean():.2f} | Mean Be My AI: {be_my_ai_products_df[question].mean():.2f}",
        end="",
    )
    if p <= 0.05:
        print(" -- STATISTICALLY SIGNIFICANT")
    else:
        print()
    print(f"U: {U}, p: {p:.4f}", end="")
    if p < 0.001:
        print("***", end="")
    elif p < 0.01:
        print("**", end="")
    elif p < 0.05:
        print("*", end="")
    elif p < 0.1:
        print(".", end="")
    print("\n")

### Differences in Percieved Impact between Be My AI and Seeing AI 

In [ ]:
print("Differences in Percieved Impact between Be My AI and Seeing AI")
print("-" * 80)

# create dataframes for each condition
be_my_ai_products_df = survey_data_df[
    survey_data_df[condition_col]
    == "Be My AI (part of Be My Eyes that does not involve human visual intepreters)"
]
seeing_ai_products_df = survey_data_df[
    survey_data_df[condition_col] == "Microsoft Seeing AI"
]

for question in image_quality_factor_cols:
    # convert to number
    seeing_ai_products_df[question] = seeing_ai_products_df[question].map(mapping_dict)
    be_my_ai_products_df[question] = be_my_ai_products_df[question].map(mapping_dict)

# remove rows where any of the image quality factor columns are empty
seeing_ai_products_df = seeing_ai_products_df[
    seeing_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
]
be_my_ai_products_df = be_my_ai_products_df[
    be_my_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
]

# print sample size
print(
    f"Be My AI: n = {len(be_my_ai_products_df)} | Seeing AI: n = {len(seeing_ai_products_df)}"
)

# calculate mean and statistical differences for how much the tool supports diagnosing issues
for question in image_quality_factor_cols:
    # filter out 1 responses ("I am not sure")
    seeing_ai_products_df = seeing_ai_products_df[seeing_ai_products_df[question] != 1]
    be_my_ai_products_df = be_my_ai_products_df[be_my_ai_products_df[question] != 1]

    U, p = mannwhitneyu(
        seeing_ai_products_df[question],
        be_my_ai_products_df[question],
        nan_policy="omit",
    )
    print(f"{question}")
    print(
        f"Mean Seeing AI: {seeing_ai_products_df[question].mean():.2f} | Mean Be My AI: {be_my_ai_products_df[question].mean():.2f}",
        end="",
    )
    if p <= 0.05:
        print(" -- STATISTICALLY SIGNIFICANT")
    else:
        print()

    print(f"p: {p:.4f}, U: {U}", end="")
    if p < 0.001:
        print("***", end="")
    elif p < 0.01:
        print("**", end="")
    elif p < 0.05:
        print("*", end="")
    elif p < 0.1:
        print(".", end="")

    # for seeing_ai_products_df, print the number of responses for each category (1-5)
    categories = [1, 2, 3, 4, 5]
    counts_seeing_ai = (
        seeing_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_be_my_ai = (
        be_my_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_df = pd.DataFrame(
        {
            "Rating": categories,
            "Seeing AI": counts_seeing_ai.values,
            "Be My AI": counts_be_my_ai.values,
        }
    )
    counts_df.loc["Total"] = counts_df.sum()
    display(counts_df)

    print("\n")

### Differences in Assessment Ability between Be My AI and Seeing AI

In [ ]:
print("Differences in Assessment Ability between Be My AI and Seeing AI")
print("-" * 80)

# create dataframes for each condition
be_my_ai_products_df = survey_data_df[
    survey_data_df[condition_col]
    == "Be My AI (part of Be My Eyes that does not involve human visual intepreters)"
]
seeing_ai_products_df = survey_data_df[
    survey_data_df[condition_col] == "Microsoft Seeing AI"
]

for question in image_assessment_factor_cols:
    # convert to number
    seeing_ai_products_df[question] = seeing_ai_products_df[question].map(mapping_dict)
    be_my_ai_products_df[question] = be_my_ai_products_df[question].map(mapping_dict)

# remove rows where any of the image quality factor columns are empty
seeing_ai_products_df = seeing_ai_products_df[
    seeing_ai_products_df[image_assessment_factor_cols].notna().all(axis=1)
]
be_my_ai_products_df = be_my_ai_products_df[
    be_my_ai_products_df[image_assessment_factor_cols].notna().all(axis=1)
]

# print sample size
print(
    f"Be My AI: n = {len(be_my_ai_products_df)} | Seeing AI: n = {len(seeing_ai_products_df)}"
)

# calculate mean and statistical differences for how much the tool supports diagnosing issues
for question in image_assessment_factor_cols:
    # filter out 1 responses ("I am not sure")
    seeing_ai_products_df = seeing_ai_products_df[seeing_ai_products_df[question] != 1]
    be_my_ai_products_df = be_my_ai_products_df[be_my_ai_products_df[question] != 1]

    U, p = mannwhitneyu(
        seeing_ai_products_df[question],
        be_my_ai_products_df[question],
        nan_policy="omit",
    )
    print(f"{question}")
    print(
        f"Mean Seeing AI: {seeing_ai_products_df[question].mean():.2f} | Mean Be My AI: {be_my_ai_products_df[question].mean():.2f}",
        end="",
    )
    if p <= 0.05:
        print(" -- STATISTICALLY SIGNIFICANT")
    else:
        print()

    print(f"p: {p:.4f}, U: {U}", end="")
    if p < 0.001:
        print("***", end="")
    elif p < 0.01:
        print("**", end="")
    elif p < 0.05:
        print("*", end="")
    elif p < 0.1:
        print(".", end="")
    print("\n")

    # for seeing_ai_products_df, print the number of responses for each category (1-5)
    categories = [1, 2, 3, 4, 5]
    counts_seeing_ai = (
        seeing_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_be_my_ai = (
        be_my_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_df = pd.DataFrame(
        {
            "Rating": categories,
            "Seeing AI": counts_seeing_ai.values,
            "Be My AI": counts_be_my_ai.values,
        }
    )
    counts_df.loc["Total"] = counts_df.sum()
    display(counts_df)

    print("\n")